In [91]:
import tkinter as tk
from tkinter import ttk
import random

In [92]:
def adicionar_texto(texto, fonte):
    global janela
    global design
    label = ttk.Label(
        janela, 
        text = texto,
        font = fonte,
    )
    return label

In [93]:
def centralizar_janela(width, height):
    global janela

    janela_width = janela.winfo_screenwidth()
    janela_height = janela.winfo_screenheight()

    pos_x = (janela_width // 2) - (width // 2)
    pos_y = (janela_height // 2) - (height // 2)

    janela.geometry(f"{width}x{height}+{pos_x}+{pos_y}")

In [94]:
def selecionar_dificuldade():
    """Cria uma série de radiobuttons ao usuário, permitindo que ele selecione a dificuldade desejada."""

    global janela
    global design
    
    def confirmar_dificuldade(selecionado, frame):
        global estado_jogo
        estado_jogo["dificuldade"] = selecionado.get()
        frame.destroy()
        texto.destroy()
        
        txt_dificuldade_selecionada = adicionar_texto(
            f"Dificuldade selecionada: {estado_jogo["dificuldade"]}",
            design["fonte-mensagem"]
        )
        
        txt_dificuldade_selecionada.pack(pady = 5)
        janela.update()
        
    #lista de dificuldades possiveis
    dificuldades = ["Fácil","Médio","Difícil","Impossível"]

    #organiza a telinha ali em cima
    texto = adicionar_texto(
        "Selecione a dificuldade:", 
        design["fonte-mensagem"]
        )
    selecionado = tk.StringVar(value="Fácil")
    texto.pack(pady = 5)

    #cria um frame para as dificuldades
    frame_dificuldade = ttk.Frame(janela)
    frame_dificuldade.pack()

    #cria os radiobuttons a partir das dificuldades
    for dificuldade in dificuldades:
        tk.Radiobutton(
            frame_dificuldade,
            text = dificuldade,
            font = design["fonte-radio"],
            variable = selecionado,
            value = dificuldade
        ).pack(pady = 2)

    #botao de confirmar
    botao = tk.Button(
        frame_dificuldade, 
        text = "Selecionar", 
        font = design["fonte-botao"],
        height = design["altura-botao"],
        width = design["largura-botao"],
        command = lambda: (
            confirmar_dificuldade(selecionado, frame_dificuldade),  
            selecionar_tema(),
            contar_erros()
        )
    )
    
    botao.pack(pady = 5)
    janela.update()

In [95]:
def contar_erros():
    global estado_jogo

    if estado_jogo['dificuldade'] == "Fácil":
        estado_jogo['erros_maximos'] = 15

    elif estado_jogo['dificuldade'] == "Médio":
        estado_jogo['erros_maximos'] = 7

    elif estado_jogo['dificuldade'] == "Difícil":
        estado_jogo['erros_maximos'] = 5

    else:
        estado_jogo['erros_maximos'] = 0

    estado_jogo['label_erros'].config(
        text = f"Erros cometidos: 0 / {estado_jogo['erros_maximos']}"
    )

In [96]:
def selecionar_tema():
    global janela
    def confirmar_tema(selecionado, frame):
        global estado_jogo
        estado_jogo["tema"] = selecionado.get()
        frame.destroy()
        texto.destroy()
    
        txt_tema_selecionado = adicionar_texto(
            f"Tema selecionado: {estado_jogo["tema"]}", 
            design["fonte-mensagem"]
        )
        
        txt_tema_selecionado.pack(pady = 5)
        janela.update()
        sortear_palavra(estado_jogo["tema"])

    temas = ["Animais", "Comidas", "Elementos", "Ilum", "Paises", "Pokemon","Profissoes","Verbos"]

    #organiza a telinha ali em cima
    texto = adicionar_texto(
        "Selecione o tema:", 
        design["fonte-mensagem"]
        )
    selecionado = tk.StringVar(value="Animais")
    texto.pack()

    #cria um frame para os rb de tema
    frame_temas= ttk.Frame(janela)
    frame_temas.pack()

    #cria os radiobuttons a partir da lista de opcoes
    for tema in temas:
        tk.Radiobutton(
            frame_temas,
            text = tema,
            variable = selecionado,
            font = design["fonte-radio"],
            value = tema
        ).pack(pady = 2)

    #cria o botao de confirmar
    botao = tk.Button(
        frame_temas, 
        text = "Selecionar", 
        font = design["fonte-botao"],
        height = design["altura-botao"],
        width = design["largura-botao"],
        command = lambda: (
            confirmar_tema(selecionado, frame_temas), 
            botao.destroy()
        )
    )

    #mostra tudo
    botao.pack()
    janela.update()

In [97]:
def sortear_palavra(tema):
    global estado_jogo
    
    tema = tema.lower()
    palavras_possiveis = []

    with open(f'temas/{tema}.txt', 'r', encoding='utf-8') as arquivo:
        for linha in arquivo:
            palavras_possiveis.append(linha.strip())

    estado_jogo["palavra_escolhida"] = random.choice(palavras_possiveis).lower()
    apresentar_palavra()

In [98]:
def apresentar_palavra():
    global estado_jogo
    global janela
    global design
    
    estado_jogo["acertos_maximos"] = len(estado_jogo["palavra_escolhida"])

    #transforma as letras da palavra em "_". 
    #desconsidera caracteres especiais e coloca eles como eles sao
    #espaços sao apenas espacos
    
    for i in estado_jogo["palavra_escolhida"]:
        # considera palavras compostas
        if (i not in estado_jogo["letras"]) and (i not in estado_jogo["letras_especiais"]):
            estado_jogo["apresentado"].append(i + " ")
            estado_jogo["acertos_maximos"] -= 1 #se nao for uma letra, nao pode ser acertado
    
        else:
            estado_jogo["apresentado"].append("_")

    estado_jogo["label_palavra"] = adicionar_texto(
        " ".join(estado_jogo["apresentado"]),
        design["fonte-mensagem"]
    )
    
    estado_jogo["label_acertos"].config(
        text = f"Acertos: 0 / {estado_jogo['acertos_maximos']}",
        font = design["fonte-mensagem"]
    )

    estado_jogo["label_acertos"].pack()
    estado_jogo["label_erros"].pack()

    estado_jogo["label_palavra"].pack(pady = 20)
    estado_jogo["label_tentativas"].pack(pady = 10)
    apresentar_entrada()
    janela.update()

In [99]:
def chutar_letra(entrada):
    global estado_jogo
    global janela
    global design

    estado_jogo["mensagem"].config(text = "")
    letra = entrada.get().strip()
    letra = letra.lower()

    #verifica se a letra é permitida
    if letra in estado_jogo["letras_tentadas"]:
        estado_jogo["mensagem"].config(text = f"Você já chutou a letra {letra}. Tente novamente!")

    elif letra == estado_jogo["palavra_escolhida"]:
        estado_jogo['acertos'] = estado_jogo['acertos_maximos']
        venceu()

    elif len(letra) != 1:
        estado_jogo["mensagem"].config(text = f"Por favor, digite apenas um caracter!")

    elif letra not in estado_jogo["letras"]:
        estado_jogo["mensagem"].config(text = f"O caracter {letra} não é válido. Tente novamente!")

    else:
        estado_jogo["letras_tentadas"].append(letra)
        acertos_locais = 0
        
        for posicao_letra, letras in enumerate(estado_jogo["palavra_escolhida"]):
                letras = letras.lower()
            
                if letra == letras:
                    acertos_locais += 1
                    acertou(letras, posicao_letra)

                if letras in estado_jogo["letras_especiais"] and letra in estado_jogo["letras_problematicas"]:
                    if letra == 'a' and (letras in ['á','ã','â','à']):
                        acertos_locais += acertou(letras, posicao_letra)
                        
                    elif letra == 'e' and (letras in ['é','ê','è']):
                        acertos_locais += acertou(letras, posicao_letra)
                                
                    elif letra == 'i' and (letras in ['í','î','ì']): 
                        acertos_locais += acertou(letras, posicao_letra)
                            
                    elif letra == 'o' and (letras in ['ó','ô','õ','ò']):
                        acertos_locais += acertou(letras, posicao_letra)
                        
                    elif letra == 'u' and (letras in ['ú','û','ù']): 
                        acertos_locais += acertou(letras, posicao_letra)
        
                    elif letra == 'c' and letras == 'ç':
                        acertos_locais += acertou(letras, posicao_letra)

        if acertos_locais == 0:
            estado_jogo['erros'] = estado_jogo['erros'] + 1
            estado_jogo["mensagem"].config(
                text = f"Eita! {letra} não está na palavra!"
                )

        else:
            estado_jogo['acertos'] = estado_jogo['acertos'] + acertos_locais
            estado_jogo["mensagem"].config(
                text = f"Acertou! A letra {letra} está na palavra!"
                )

        if estado_jogo['erros'] > estado_jogo['erros_maximos']:
            perdeu()

        if estado_jogo['acertos'] >= estado_jogo['acertos_maximos']:
            venceu()
            
        estado_jogo["label_tentativas"].config(
            text = f"Letras tentadas: {' '.join(estado_jogo['letras_tentadas'])}"
        )

        estado_jogo['label_acertos'].config(
            text = f"Acertos: {estado_jogo['acertos']} / {estado_jogo['acertos_maximos']}"
            )
        
        estado_jogo['label_erros'].config(
            text = f"Erros cometidos: {estado_jogo['erros']} / {estado_jogo['erros_maximos']}"
        )

    estado_jogo['label_acertos'].update()
    estado_jogo['label_erros'].update()
    estado_jogo["mensagem"].update()
    janela.update()

In [100]:
def perdeu():
    global estado_jogo
    global design
    
    limpar_tela()
    adicionar_texto(
        "☠️VOCÊ PERDEU!☠️", 
        design["fonte-titulo"]
        ).pack(pady = (150, 50))
    
    adicionar_texto(
        f"A palavra era: {estado_jogo['palavra_escolhida']}",
        design["fonte-mensagem"]
    ).pack()

    tk.Button(
        janela,
        text="Jogar Novamente",
        command=reiniciar_jogo,
        font = design["fonte-botao"],
        height = design["altura-botao"],
        width = design["largura-botao"]
    ).pack(pady = (10, 0))

    tk.Button(
        janela,
        text="Sair",
        command=janela.destroy,
        font = design["fonte-botao"],
        height = design["altura-botao"],
        width = design["largura-botao"]
    ).pack(pady = (10, 0))

def venceu():
    global estado_jogo
    global design
    limpar_tela()
    adicionar_texto(
        "🎊PARABÉNS! VOCÊ VENCEU!🎊", 
        design["fonte-titulo"]
        ).pack(pady = (150, 50))

    adicionar_texto(
        f"A palavra era: {estado_jogo['palavra_escolhida']}",
        design["fonte-mensagem"]
    ).pack()

    tk.Button(
        janela,
        text="Jogar Novamente",
        command=reiniciar_jogo,
        font = design["fonte-botao"],
        height = design["altura-botao"],
        width = design["largura-botao"]
    ).pack(pady = (10, 0))

    tk.Button(
        janela,
        text="Sair",
        command=janela.destroy,
        font = design["fonte-botao"],
        height = design["altura-botao"],
        width = design["largura-botao"]
    ).pack(pady = (10, 0))

In [101]:
def limpar_tela():
    global janela

    for widget in janela.winfo_children():
        widget.destroy()

In [102]:
def reiniciar_jogo():
    global estado_jogo

    limpar_tela()
    titulo = adicionar_texto(
        "-- JOGO DA FORCA --", 
        design["fonte-titulo"]
        )
    
    titulo.pack()
    
    estado_jogo = {
        "letras": ['a','b','c','d','e','f','g','h','i','j','k','l','m',
                    'n','o','p','q','r','s','t','u','v','w','x','y','z'],
        
        "letras_especiais": ['á','ã','â','à','é','ê','è','í','î','ì','ó','ô','õ','ò','ú','û','ù','ç'],
        "letras_problematicas": ['a', 'e', 'i', 'o', 'u', 'c'],
        
        "dificuldade": "Fácil",
        "tema": "Animais",
        "palavra_escolhida":"0", # se tudo der errado e a palavra não ser sorteada, o 0 aparece
        "label_palavra": None,
        "apresentado": [],
        
        "erros":0,
        "erros_maximos": 0,
        "label_erros": (adicionar_texto(
            f"Erros cometidos: 0", 
            design["fonte-mensagem"])
            ),

        "acertos":0,
        "label_acertos": (adicionar_texto(
            f"Acertos feitos: 0", 
            design["fonte-mensagem"])
            ),

        "acertos_maximos": 0, # idem
        
        "letra_chutada":"0",
        "entrada": '',

        "label_tentativas": adicionar_texto(
            "Letras tentadas:", 
            design["fonte-mensagem"]
            ),
        "letras_tentadas":[],

        "mensagem": adicionar_texto(
            f"Erros cometidos: 0", 
            design["fonte-mensagem"]
            )
    
    }

    selecionar_dificuldade()

In [103]:
def acertou(letra, posicao):
    global estado_jogo

    estado_jogo["apresentado"][posicao] = letra
    estado_jogo["label_palavra"].config(
        text=" ".join(estado_jogo["apresentado"])
    )
    return 1

In [104]:
def apresentar_entrada():
    global design
    global janela
    global estado_jogo
    
    estado_jogo["entrada"] = tk.Entry(janela, font = design["fonte-mensagem"])
    estado_jogo["entrada"].pack()

    botao = tk.Button(
        janela,
        text = "Chutar",
        font = design["fonte-botao"],
        height = design["altura-botao"],
        width = design["largura-botao"],
        command = lambda:(
            chutar_letra(estado_jogo["entrada"]),
            estado_jogo["entrada"].delete(0, tk.END)
            )
    )
    
    botao.pack(pady = 20)
    estado_jogo["mensagem"].pack()
    janela.update()

In [105]:
def iniciar_jogo():
    global janela
    
    #cenetraliza a janela
    width = 800
    height = 500
    centralizar_janela(width, height)
    titulo = adicionar_texto(
        "-- JOGO DA FORCA --", 
        design["fonte-titulo"]
        )
    
    #configura botão inicial
    botao = tk.Button(
        janela,
        text = "JOGAR",    
        font = design["fonte-botao"],
        height = design["altura-botao"],
        width = design["largura-botao"],
        command = lambda: (
            botao.destroy(),
            selecionar_dificuldade() #seleciona a dificuldade do jogo
            )
        )
    
    #mostra todo mundo
    titulo.pack()
    botao.pack(pady = 50)
    janela.mainloop()


janela = janela = tk.Tk()
style = ttk.Style()
style.theme_use("vista")

design = {
    "fonte-titulo": ("Arial", 26, "bold"),
    "fonte-mensagem": ("Arial", 12),
    "fonte-radio": ("Arial", 12),
    "fonte-botao": ("Arial", 12),

    "altura-botao": 1,
    "largura-botao":15
}

estado_jogo = {
    "letras": ['a','b','c','d','e','f','g','h','i','j','k','l','m',
                'n','o','p','q','r','s','t','u','v','w','x','y','z'],
    
    "letras_especiais": ['á','ã','â','à','é','ê','è','í','î','ì','ó','ô','õ','ò','ú','û','ù','ç'],
    "letras_problematicas": ['a', 'e', 'i', 'o', 'u', 'c'],
    
    "dificuldade": "Fácil",
    "tema": "Animais",
    "palavra_escolhida":"0", # se tudo der errado e a palavra não ser sorteada, o 0 aparece
    "label_palavra": None,
    "apresentado": [],
    
    "erros":0,
    "erros_maximos": 0,
    "label_erros": (adicionar_texto(
        f"Erros cometidos: 0", 
        design["fonte-mensagem"])
        ),

    "acertos":0,
    "label_acertos": (adicionar_texto(
        f"Acertos feitos: 0", 
        design["fonte-mensagem"])
        ),

    "acertos_maximos": 0, # idem
    
    "letra_chutada":"0",
    "entrada": '',

    "label_tentativas": adicionar_texto(
        "Letras tentadas:", 
        design["fonte-mensagem"]
        ),
    "letras_tentadas":[],

    "mensagem": adicionar_texto(
        f"Erros cometidos: 0", 
        design["fonte-mensagem"]
        )
    
    }

iniciar_jogo()

Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Program Files\Python313\Lib\tkinter\__init__.py", line 2074, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\heloisa2610016\AppData\Local\Temp\ipykernel_13016\2314798071.py", line 16, in <lambda>
    chutar_letra(estado_jogo["entrada"]),
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\heloisa2610016\AppData\Local\Temp\ipykernel_13016\4179699912.py", line 72, in chutar_letra
    estado_jogo["label_tentativas"].config(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        text = f"Letras tentadas: {' '.join(estado_jogo['letras_tentadas'])}"
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Program Files\Python313\Lib\tkinter\__init__.py", line 1828, in configure
    return self._configure('configure', cnf, kw)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Program Files\Python313\Lib\tkinter\__init__.py", 